# 🎙️ IndexTTS-2.5 Audiobook Generation (Google Colab CUDA Runner)

A robust, resumable audiobook generation environment powered by **Google Colab (CUDA GPU)** and the **`index-tts-audiobook`** orchestration pipeline.

### Key Architecture & Features:
- **Google Drive Persistence**: Models, voice prompts, markdown chapters, and final audiobooks are stored directly in your Google Drive. Nothing is lost when a Colab session disconnects.
- **Resume Safety**: The pipeline renders chunk-by-chunk with SHA-256 manifests. If Colab disconnects or times out, re-running the render cell **instantly resumes** from the last completed chunk.
- **One-Time Checkpoint Download**: IndexTTS-2.5 weights (~4.3 GB) are downloaded once to Google Drive and reused across all subsequent sessions.
- **Hardware**: Recommended GPU: **T4** (Free tier) or **L4 / A100** (Colab Pro).

> 💡 **Setup**: Make sure hardware acceleration is set to GPU:  
> `Runtime` -> `Change runtime type` -> select `T4 GPU` (or `L4` / `A100`).

## 1. Check GPU & CUDA Environment
Confirm that an NVIDIA GPU is allocated and ready.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    raise SystemError("❌ GPU not detected! Please go to Runtime -> Change runtime type and select T4 or A100 GPU.")

## 2. Mount Google Drive & Establish Workspace
This mounts Google Drive and sets up the workspace directory structure:
```
MyDrive/audiobook-workspace/
├── checkpoints/IndexTTS-2.5/   # Cached model weights
├── prompts/                  # Speaker reference audio (e.g. narrator.wav)
├── scripts/                  # Narration manuscripts (.md)
├── output/                   # Rendered chapters (.wav) & manifests
└── config/                   # Pipeline configuration (.toml)
```

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/audiobook-workspace')
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints' / 'IndexTTS-2.5'
PROMPTS_DIR = DRIVE_ROOT / 'prompts'
SCRIPTS_DIR = DRIVE_ROOT / 'scripts'
OUTPUT_DIR = DRIVE_ROOT / 'output'
CONFIG_DIR = DRIVE_ROOT / 'config'

for path in [CHECKPOINTS_DIR, PROMPTS_DIR, SCRIPTS_DIR, OUTPUT_DIR, CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("✅ Google Drive workspace initialized:")
print(f"  Checkpoints: {CHECKPOINTS_DIR}")
print(f"  Prompts:     {PROMPTS_DIR}")
print(f"  Scripts:     {SCRIPTS_DIR}")
print(f"  Output:      {OUTPUT_DIR}")
print(f"  Config:      {CONFIG_DIR}")

## 3. Clone Upstream IndexTTS & Install Audiobook Pipeline
We clone the official upstream IndexTTS into Colab local storage (`/content/index-tts`) and install dependencies along with `index-tts-audiobook`.

In [ ]:
import os
%cd /content

if not os.path.exists("/content/index-tts"):
    !git clone --depth 1 https://github.com/index-tts/index-tts.git /content/index-tts

%cd /content/index-tts
# 1. Disable and remove pre-installed tensorflow to prevent protobuf conflicts
!pip uninstall -y -q tensorflow

# 2. Install IndexTTS dependencies with exact versions
!pip install -q -e .
!pip install -q -U "transformers==4.52.1" "accelerate==1.8.1" "tokenizers==0.21.0"
!pip install -q -U openai-whisper descript-audiotools cn2an g2p-en WeTextProcessing
!pip install -q -U munch omegaconf einops json5 textstat pydub sentencepiece safetensors librosa jieba fugashi unidic-lite
!pip install -q opencc-python-reimplemented soundfile huggingface_hub modelscope

# 3. Install latest audiobook pipeline with USE_TF=0 support
!pip install -q --upgrade "index-tts-audiobook[indextts] @ git+https://github.com/HOWARD1021/index-tts-audiobook.git"
import fugashi, unidic_lite
print("✅ Japanese G2P dependencies are installed.")

print("✅ Runtimes and dependencies installed successfully!")

## 4. Download Model Checkpoints (One-time Setup)
Check whether the IndexTTS-2 checkpoints already exist in Google Drive. If not, download them via Hugging Face. Subsequent sessions will detect the existing files and complete in seconds.

In [ ]:
from pathlib import Path
import os
from huggingface_hub import snapshot_download

DRIVE_ROOT = Path('/content/drive/MyDrive/audiobook-workspace')
CHECKPOINTS_DIR = globals().get('CHECKPOINTS_DIR', DRIVE_ROOT / 'checkpoints' / 'IndexTTS-2.5')
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

required_files = ['config.yaml', 'codec.pth', 'gpt.pth', 's2mel.pth', 'wav2vec2bert_stats.pt']
files_exist = all((CHECKPOINTS_DIR / f).exists() for f in required_files)

if files_exist:
    print(f'✅ Checkpoints already exist in Google Drive at:\n  {CHECKPOINTS_DIR}')
else:
    print('⏳ Checkpoints not found. Downloading IndexTTS-2.5 (~4.3 GB) to Google Drive...')
    snapshot_download(
        repo_id='IndexTeam/IndexTTS-2.5',
        local_dir=str(CHECKPOINTS_DIR),
        local_dir_use_symlinks=False,
    )
    print('✅ IndexTTS-2.5 download completed!')

print('\nCheckpoint files:')
for f in sorted(os.listdir(CHECKPOINTS_DIR)):
    if not f.startswith('.'):
        print(f'  - {f}')


## 5. Configure CUDA Profile
Generate a `colab-cuda.toml` configuration setting `device = "cuda"` and optimal audio parameters.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/audiobook-workspace')
CONFIG_DIR = globals().get('CONFIG_DIR', DRIVE_ROOT / 'config')
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

colab_config_path = CONFIG_DIR / 'colab-cuda.toml'
config_content = """[defaults]
language = "ZH"
device = "cuda"
max_chunk_chars = 400
max_text_tokens_per_segment = 100
interval_silence_ms = 250
inter_chunk_pause_ms = 450
emotion_span_pause_ms = 80
text_normalization = true
use_random = false
use_qwen_emo = false
sample_rate = 22050
channels = 1
max_seconds_per_char = 0.8
max_mel_tokens = 800
temperature = 1.0
top_k = 30
top_p = 0.8
repetition_penalty = 10.0
speed = 1.0
seed = 42
memory_limit_gb = 16.0

[defaults.emotion]
vector = [0.30, 0.0, 0.0, 0.0, 0.0, 0.0, 0.15, 0.35]
alpha = 1.0
bold_vector = [0.45, 0.0, 0.0, 0.0, 0.0, 0.0, 0.20, 0.10]
bold_alpha = 1.0
italic_vector = [0.15, 0.0, 0.0, 0.0, 0.0, 0.20, 0.0, 0.45]
italic_alpha = 1.0
"""
colab_config_path.write_text(config_content.strip(), encoding='utf-8')
print(f'✅ Configuration written to: {colab_config_path}')


## 6. Prepare Narration Script & Prompt Audio
1. **Speaker reference**: Place your narrator WAV (5–15 seconds of clean speech) into `MyDrive/audiobook-workspace/prompts/` (e.g. `narrator.wav`).
2. **Script**: Place your chapter markdown file into `MyDrive/audiobook-workspace/scripts/` (e.g. `chapter-01.md`).

*(The cell below creates a starter sample if no script is present yet).* 

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/audiobook-workspace')
PROMPTS_DIR = globals().get('PROMPTS_DIR', DRIVE_ROOT / 'prompts')
SCRIPTS_DIR = globals().get('SCRIPTS_DIR', DRIVE_ROOT / 'scripts')
for p in [PROMPTS_DIR, SCRIPTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

sample_script = SCRIPTS_DIR / 'sample-chapter.md'
if not sample_script.exists():
    sample_text = """# 第一章：啟程

這是一個寧靜的早晨，陽光穿透薄霧，灑在青石街道上。旅人背起行囊，準備迎接未知的冒險。

**「這條路將會通向何方？」** 他心中自問，步伐卻顯得堅定無比。

林間的微風帶著淡淡的泥土芳香，遠方的山巒在晨光中若隱若現。這段旅程，才剛剛開始。
"""
    sample_script.write_text(sample_text, encoding='utf-8')
    print(f'Created sample script: {sample_script}')

prompt_wavs = list(PROMPTS_DIR.glob('*.wav'))
if not prompt_wavs:
    print('\n⚠️ No .wav files found in prompts directory!')
    print(f'👉 Please upload your narrator reference WAV into: {PROMPTS_DIR}')
else:
    print(f'\n✅ Found speaker prompt(s): {[p.name for p in prompt_wavs]}')


## 7. Script Preparation & Chunk Planning
Run `audiobook prepare` to convert Traditional Chinese into Simplified Chinese for the Mandarin model, remove non-spoken markup, and plan chunk boundaries.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/audiobook-workspace')
SCRIPTS_DIR = globals().get('SCRIPTS_DIR', DRIVE_ROOT / 'scripts')
CONFIG_DIR = globals().get('CONFIG_DIR', DRIVE_ROOT / 'config')
colab_config_path = globals().get('colab_config_path', CONFIG_DIR / 'colab-cuda.toml')

raw_script_path = str(SCRIPTS_DIR / 'sample-chapter.md')
prepared_script_path = str(SCRIPTS_DIR / 'sample-chapter-simplified.md')

# 1. Preprocess manuscript
!audiobook prepare --input "{raw_script_path}" --output "{prepared_script_path}"

# 2. Inspect planned chunks
!audiobook plan --script "{prepared_script_path}" --max-chars 400 --config "{colab_config_path}"


## 8. Render Chapter with IndexTTS-2.5 on CUDA
Synthesize audio on GPU.  
- **Resumable**: Each chunk is saved immediately. If connection drops, simply re-run this cell to resume without re-synthesizing completed chunks.
- **Local Emotion**: Bold text markdown elements are assigned distinct expressive emotion vectors automatically.

In [ ]:
from pathlib import Path
import os
import subprocess
from huggingface_hub import snapshot_download

# 1. 工作區路徑設定
DRIVE_ROOT = Path("/content/drive/MyDrive/audiobook-workspace")
CHECKPOINTS_DIR = globals().get("CHECKPOINTS_DIR", DRIVE_ROOT / "checkpoints" / "IndexTTS-2.5")
PROMPTS_DIR = globals().get("PROMPTS_DIR", DRIVE_ROOT / "prompts")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", DRIVE_ROOT / "scripts")
OUTPUT_DIR = globals().get("OUTPUT_DIR", DRIVE_ROOT / "output")
CONFIG_DIR = globals().get("CONFIG_DIR", DRIVE_ROOT / "config")
colab_config_path = globals().get("colab_config_path", CONFIG_DIR / "colab-cuda.toml")
prepared_script_path = globals().get("prepared_script_path", SCRIPTS_DIR / "sample-chapter-simplified.md")

for path in [CHECKPOINTS_DIR, PROMPTS_DIR, SCRIPTS_DIR, OUTPUT_DIR, CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# 2. 自動檢查並補齊 IndexTTS-2.5 完整權重
required_files = ["config.yaml", "codec.pth", "gpt.pth", "s2mel.pth", "wav2vec2bert_stats.pt"]
if not all((CHECKPOINTS_DIR / filename).exists() for filename in required_files):
    print("⏳ 正在補齊 IndexTTS-2.5 官方模型權重（約 4.3 GB）...")
    snapshot_download(
        repo_id="IndexTeam/IndexTTS-2.5",
        local_dir=str(CHECKPOINTS_DIR),
        local_dir_use_symlinks=False,
    )
    print("✅ IndexTTS-2.5 完整模型權重已下載完成！")
else:
    print("✅ IndexTTS-2.5 模型權重已就緒。")

# 3. 確認聲音檔
prompt_wavs = list(PROMPTS_DIR.glob("*.wav"))
if not prompt_wavs:
    raise FileNotFoundError(f"請在 Google Drive 的 {PROMPTS_DIR} 放入 voice.wav 參考音訊！")

selected_prompt = prompt_wavs[0]
output_wav = OUTPUT_DIR / "sample-chapter.wav"
print(f"🎙️ 參考聲音: {selected_prompt.name}")
print(f"🎯 輸出目標: {output_wav}")
print("🚀 開始在 CUDA GPU 進行合成...")

# 4. 啟動 GPU 合成
render_env = os.environ.copy()
render_env["USE_TF"] = "0"
subprocess.run(
    [
        "audiobook", "render",
        "--backend", "indextts-2.5",
        "--script", str(prepared_script_path),
        "--output", str(output_wav),
        "--project-root", "/content/index-tts",
        "--model-dir", str(CHECKPOINTS_DIR),
        "--prompt", str(selected_prompt),
        "--config", str(colab_config_path),
        "--device", "cuda",
    ],
    env=render_env,
    check=True,
)


## 9. Audio Validation & In-Browser Audio Player
Verify that the generated audio passes format checks (mono PCM 16-bit 22,050 Hz, finite samples) and listen to the result inline.

In [ ]:
from pathlib import Path
import json
import soundfile as sf
from IPython.display import Audio, display

DRIVE_ROOT = Path('/content/drive/MyDrive/audiobook-workspace')
OUTPUT_DIR = globals().get('OUTPUT_DIR', DRIVE_ROOT / 'output')
CONFIG_DIR = globals().get('CONFIG_DIR', DRIVE_ROOT / 'config')
colab_config_path = globals().get('colab_config_path', CONFIG_DIR / 'colab-cuda.toml')

output_wav = OUTPUT_DIR / 'sample-chapter.wav'
manifest_path = OUTPUT_DIR / 'sample-chapter.manifest.json'

# Validate audio with pipeline quality gates
!audiobook validate --wav "{output_wav}" --backend indextts-2.5 --config "{colab_config_path}"

if manifest_path.exists():
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
    chunks = manifest.get('chunks', [])
    duration = manifest.get('final_duration_seconds', 0)
    print(f'\n📊 Chapter Summary:')
    print(f'  Status:          {manifest.get("status")}')
    print(f'  Chunks rendered: {len(chunks)}')
    print(f'  Total Duration:  {duration:.2f}s ({duration/60:.2f} mins)')

# Play audio
if output_wav.exists():
    data, sr = sf.read(str(output_wav))
    print(f'\n▶️ Playback ({sr} Hz, {len(data)} samples):')
    display(Audio(data, rate=sr))


## 10. (Optional) Batch Render Multiple Chapters
Iterate through all markdown chapters in `scripts/` (e.g. `chapter-01.md`, `chapter-02.md`, ...) and render them automatically.

In [ ]:
from pathlib import Path
import os
import subprocess
from huggingface_hub import snapshot_download

DRIVE_ROOT = Path("/content/drive/MyDrive/audiobook-workspace")
CHECKPOINTS_DIR = globals().get("CHECKPOINTS_DIR", DRIVE_ROOT / "checkpoints" / "IndexTTS-2.5")
PROMPTS_DIR = globals().get("PROMPTS_DIR", DRIVE_ROOT / "prompts")
SCRIPTS_DIR = globals().get("SCRIPTS_DIR", DRIVE_ROOT / "scripts")
OUTPUT_DIR = globals().get("OUTPUT_DIR", DRIVE_ROOT / "output")
CONFIG_DIR = globals().get("CONFIG_DIR", DRIVE_ROOT / "config")
colab_config_path = globals().get("colab_config_path", CONFIG_DIR / "colab-cuda.toml")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 確保 IndexTTS-2.5 完整權重存在
required_files = ["config.yaml", "codec.pth", "gpt.pth", "s2mel.pth", "wav2vec2bert_stats.pt"]
if not all((CHECKPOINTS_DIR / filename).exists() for filename in required_files):
    print("⏳ 正在下載 IndexTTS-2.5 完整模型權重至 Google Drive...")
    snapshot_download(
        repo_id="IndexTeam/IndexTTS-2.5",
        local_dir=str(CHECKPOINTS_DIR),
        local_dir_use_symlinks=False,
    )

# 1. 檢查 scripts 目錄下所有檔案
print(f"📂 檢查目錄: {SCRIPTS_DIR}")
all_files = sorted(SCRIPTS_DIR.iterdir())
print(f"目前 scripts/ 目錄下共有 {len(all_files)} 個檔案/項目：")
for file_path in all_files:
    print(f"  - {file_path.name}")

# 2. 篩選章節稿件 (支援 .md 與 .txt，自動排除 sample-chapter)
chapters = sorted(
    path for path in SCRIPTS_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".md", ".txt"}
    and not path.name.startswith("sample-")
)

prompt_wavs = list(PROMPTS_DIR.glob("*.wav"))
if not prompt_wavs:
    raise FileNotFoundError(f"Missing prompt WAV in {PROMPTS_DIR}")
selected_prompt = prompt_wavs[0]

render_env = os.environ.copy()
render_env["USE_TF"] = "0"

if not chapters:
    print("⚠️ 尚未找到章節檔案！請確認 scripts 資料夾中有 .md 或 .txt 章節稿。")
else:
    print(f"📚 找到 {len(chapters)} 個章節檔案準備批次合成:")
    for chapter in chapters:
        print(f"  👉 {chapter.name}")

    for chapter in chapters:
        clean_stem = chapter.stem.replace("-zh-simplified", "").replace("-simplified", "")
        output_wav = OUTPUT_DIR / f"{clean_stem}.wav"

        # 簡體稿直接使用；其他稿件先做繁簡轉換與文字整理。
        if "-simplified" in chapter.stem:
            render_script = chapter
        else:
            render_script = SCRIPTS_DIR / f"{clean_stem}-simplified.md"
            subprocess.run(
                ["audiobook", "prepare", "--input", str(chapter), "--output", str(render_script)],
                check=True,
            )

        print()
        print("=" * 55)
        print(f"🚀 正在批次合成: {clean_stem} -> {output_wav.name}")
        print("=" * 55)

        subprocess.run(
            [
                "audiobook", "render",
                "--backend", "indextts-2.5",
                "--script", str(render_script),
                "--output", str(output_wav),
                "--project-root", "/content/index-tts",
                "--model-dir", str(CHECKPOINTS_DIR),
                "--prompt", str(selected_prompt),
                "--config", str(colab_config_path),
                "--device", "cuda",
            ],
            env=render_env,
            check=True,
        )
        subprocess.run(
            [
                "audiobook", "validate",
                "--wav", str(output_wav),
                "--backend", "indextts-2.5",
                "--config", str(colab_config_path),
            ],
            check=True,
        )

    print("🎉 所有章節批次合成完成！")
